# Implementation


In [ ]:
!pip install langchain
!pip install langchain_text_splitters
!pip install openai
!pip install -U "langchain[google-genai]"
!pip install pydantic
!pip install langchain-groq
!pip install neo4j
!pip install networkx
!pip install pyvis
!pip install cdlib
!pip install leidenalg
#!pip install scispacy spacy
#!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz

In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chat_models import init_chat_model
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel,Field
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq
from neo4j import GraphDatabase
import networkx as nx
import matplotlib.pyplot as plt
from pyvis.network import Network
from IPython.display import display, HTML
from cdlib import algorithms
from typing import List, Literal
import time
from tqdm import tqdm 
from langchain_core.output_parsers import PydanticOutputParser
#import spacy
#from scispacy.linking import EntityLinker
groq_key=[]
URI="" 
AUTH=""


In [ ]:
os.environ["GROQ_API_KEY"]=groq_key[2]

In [ ]:
text=""

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""]
)
chunks=splitter.split_text(text)

In [ ]:
from typing import List, Literal
from pydantic import BaseModel, Field

EntityType = Literal[
    # CLASS A: BIOLOGICAL AGENTS
    "PATHOGEN",          # Viruses, Bacteria, Fungi (e.g., H3N2, E. coli)
    "HOST",              # Patients, Animals, Demographics (e.g., Refugee patients, Children, Ferrets)

    # CLASS B: MOLECULAR REALITY (Crucial for your data)
    "GENETIC_ENTITY",    # Genes, RNA, DNA, Primers, Amplicons (e.g., HA Gene, PCR Primers)
    "MOLECULAR_VARIANT", # Mutations, Substitutions, Strains, Isolates (e.g., K145N, Nepal Isolates)
    "CHEMICAL",          # Drugs, Proteins, Reagents, Antibodies (e.g., Lisinopril, HA Protein)

    # CLASS C: CLINICAL REALITY
    "CONDITION",         # Diseases, Symptoms, Syndromes (e.g., Influenza, Fever)
    "PHENOMENON",        # Biological processes (e.g., Genetic Drift, Antigenic Variation)

    # CLASS D: INTERVENTION & CONTEXT
    "PROCEDURE",         # Tests, Assays, Sequencing, Vaccinations (e.g., PCR, HI Assay)
    "DEVICE",            # Machines, Kits (e.g., Magnapure LX)
    "LOCATION",          # Geographic or Anatomical (e.g., Nepal, Throat)
    "METRIC",            # Measurements, Titers (e.g., 1:320, 4-fold increase)
    "OTHER"              # Strict fallback
]

RelationType = Literal[
    "CAUSES",            # PATHOGEN/VARIANT -> CONDITION (H3N2 causes Influenza)
    "TREATS",            # CHEMICAL/PROCEDURE -> CONDITION (Tamiflu treats Influenza)
    "PREVENTS",          # VACCINE/CHEMICAL -> CONDITION (Vaccine prevents Flu)

    "EXHIBITS",          # PATHOGEN -> VARIANT (H3N2 exhibits K145N mutation)
    "DETECTS",           # PROCEDURE -> PATHOGEN/GENETIC_ENTITY (PCR detects RNA)
    "TARGETS",           # CHEMICAL -> GENETIC_ENTITY/PATHOGEN (Primers target HA Gene)

    "ORIGINATES_FROM",   # VARIANT -> LOCATION/HOST (Nepal Isolates originate from Nepal)
    "DIFFERS_FROM",      # VARIANT -> VARIANT (Strain A differs from Strain B)

    "HAS_CONTEXT",       # ENTITY -> LOCATION/METRIC/TIME (General linker)
    "ASSOCIATED_WITH"    # Fallback for weak correlations
]

class Entity(BaseModel):
    name: str = Field(description="The precise scientific name. For 'Nepal Isolates', extract 'Isolates' as the name and 'Nepal' as a separate Location node.")
    type: EntityType = Field(description="The strict scientific category.")

class Relation(BaseModel):
    source_entity: Entity = Field(description="The subject of the relation")
    relation_type: RelationType = Field(description="The scientific verb connecting them")
    target_entity: Entity = Field(description="The object of the relation")
    evidence: str = Field(description="The EXACT quote from the text supporting this.")
    confidence: float = Field(description="Confidence score (0.0-1.0).")

class MedicalGraph(BaseModel):
    relations: List[Relation]

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
import time
from tqdm import tqdm 
from langchain_core.output_parsers import PydanticOutputParser

#llm = ChatGoogleGenerativeAI(
 #   model="gemini-2.5-flash-lite",
  #  temperature=0
#)
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    max_retries=2,
)

structured_llm = llm.with_structured_output(MedicalGraph)

system_prompt = """You are an expert Medical Knowledge Graph Engineer.
Your goal is to extract a highly precise, scientifically accurate Knowledge Graph from the provided text.

### CORE INSTRUCTION: DECONSTRUCTION OVER LABELING
Medical text often compresses complex relationships into single noun phrases.
Do NOT label complex phrases as single nodes. Instead, DECONSTRUCT them into atomic components.

Examples of Deconstruction:
- "Metastatic Lung Cancer" -> "Lung Cancer" (CONDITION) + "Metastasis" (PHENOMENON).
- "Nepal Isolates" -> "Isolates" (MOLECULAR_VARIANT) + "Nepal" (LOCATION).
- "PCR Primers" -> "Primers" (GENETIC_ENTITY) + "PCR" (PROCEDURE).

### STRICT ENTITY GUIDELINES
1. PATHOGEN: Only the active infectious agent (e.g., "H3N2", "S. aureus"). Do not confuse with the Disease.
2. CONDITION: The passive disease or symptom (e.g., "Influenza", "Pneumonia").
3. CHEMICAL: Includes drugs, proteins, and lab reagents.
4. MOLECULAR_VARIANT: Use this for specific strains, mutations, or receptor statuses (e.g., "Delta Variant", "HER2+").
5. GENETIC_ENTITY: Use this for Genes, RNA, DNA, and Primers.
6. LOCATION: Use this for both Geography (Countries) and Anatomy (Body parts).

### RELATIONSHIP USAGE RULES
- Use 'EXHIBITS' to link a Subject to its Properties or Variants (e.g., Virus -> EXHIBITS -> Mutation).
- Use 'TARGETS' for Mechanism of Action (e.g., Drug -> TARGETS -> Protein).
- Use 'ORIGINATES_FROM' to link samples/variants to their source (e.g., Tumor -> ORIGINATES_FROM -> Tissue).
- Use 'HAS_CONTEXT' to link Entities to Metrics, Time, or Locations (e.g., Patient -> HAS_CONTEXT -> Age 60).
- Use 'DETECTS' for Diagnostic tools finding a Condition or Pathogen.

### OUTPUT LOGIC
- If the text mentions "Resistance to Tamiflu", extract:
  (Tamiflu: CHEMICAL) -[TREATS]-> (Influenza: CONDITION)
  (Influenza: CONDITION) -[EXHIBITS]-> (Resistance: PHENOMENON)
  (Resistance: PHENOMENON) -[ASSOCIATED_WITH]-> (Tamiflu: CHEMICAL)

Analyze the text deeply. Bias towards granular, scientifically precise nodes.
"""

few_shot_examples = """
Input: "Patient diagnosed with hypertension. Prescribed Lisinopril 10mg."
Output: [
  {"source": "Lisinopril", "relation": "TREATS", "target": "Hypertension", "evidence": "Prescribed Lisinopril 10mg"}
]

Input: "Patient denies chest pain but reports shortness of breath."
Output: [
  {"source": "Patient", "relation": "ASSOCIATED_WITH", "target": "Shortness of Breath", "evidence": "reports shortness of breath"}
  // Note: We deliberately ignored Chest Pain because it was denied.
]
"""

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "Here are some examples of correct extraction: {examples}"),
    ("human", "Analyze this clinical note: {text}")
])


chain = prompt_template | structured_llm

final_results = []
errors = []

print(f"Starting extraction on {len(chunks)} chunks...")

for i, chunk in tqdm(enumerate(chunks), total=len(chunks)):
    try:
        result = chain.invoke({
            "examples": few_shot_examples,
            "text": chunk
        })

        final_results.append(result.model_dump())

    except Exception as e:
        print(f"Error on chunk {i}: {e}")
        errors.append({"chunk_index": i, "error": str(e)})
    time.sleep(10)

import json

with open("medical_graph_output.json", "w") as f:
    json.dump(final_results, f, indent=2)

print(f"Completed! Extracted {len(final_results)} graphs. Errors: {len(errors)}")

In [ ]:

merge_query="""
MERGE (e:Entity{name:$name1})
ON CREATE
  SET e.type=$type1

MERGE (f:Entity{name:$name2})
ON CREATE
  SET f.type=$type2

MERGE (e)-[r:RELATIONSHIP]->(f)
"""
relationships=[
    "CAUSES",            # PATHOGEN/VARIANT -> CONDITION (H3N2 causes Influenza)
    "TREATS",            # CHEMICAL/PROCEDURE -> CONDITION (Tamiflu treats Influenza)
    "PREVENTS",          # VACCINE/CHEMICAL -> CONDITION (Vaccine prevents Flu)

    "EXHIBITS",          # PATHOGEN -> VARIANT (H3N2 exhibits K145N mutation)
    "DETECTS",           # PROCEDURE -> PATHOGEN/GENETIC_ENTITY (PCR detects RNA)
    "TARGETS",           # CHEMICAL -> GENETIC_ENTITY/PATHOGEN (Primers target HA Gene)

    "ORIGINATES_FROM",   # VARIANT -> LOCATION/HOST (Nepal Isolates originate from Nepal)
    "DIFFERS_FROM",      # VARIANT -> VARIANT (Strain A differs from Strain B)

    "HAS_CONTEXT",       # ENTITY -> LOCATION/METRIC/TIME (General linker)
    "ASSOCIATED_WITH"    # Fallback for weak correlations
]
import json
x=json.dumps(final_results)
x=json.loads(x)

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()
    print("Connection established.")
    #record,summary,key=driver.execute_query("MATCH (p:Person) RETURN p.name",database_="neo4j")  ---------------->>>> for read operation
    record,summary,key=driver.execute_query("MATCH(n) DETACH DELETE n",database_="neo4j")
    for i in x:
      for j in i['relations']:
        source_name=j['source_entity'].get('name')
        source_type=j['source_entity'].get('type')
        target_name=j['target_entity'].get('name')
        target_type=j['target_entity'].get('type')
        relationship=j['relation_type']
        if not (source_name and target_name and relationship):
                continue
        if relationship in relationships:
          final_query=merge_query.replace("RELATIONSHIP",relationship)
          summary=driver.execute_query(final_query,name1=source_name,type1=source_type,name2=target_name,type2=target_type,database_="neo4j")
        else:
          print("----LLM HALLUCINATED AND NOT FOLLOWED ORDERS-----------------------------------------------------")


In [ ]:
g=nx.Graph()

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()
    print("Connection established.")
    records,summary,key=driver.execute_query("MATCH (n:Entity)-[r]->(m:Entity) RETURN n,r,m")
for record in records:
  x=record.data()
  source=x.get("n")
  source_nxname=source["name"]
  source_nxtype=source["type"]
  rel=x.get("r")
  rel=rel[1]
  target=x.get("m")
  target_nxname=target["name"]
  target_nxtype=target["type"]
  g.add_node(source_nxname)
  g.add_node(target_nxname)
  g.add_edge(source_nxname,target_nxname)
  g.nodes[source_nxname]["type"]=source_nxtype
  g.nodes[target_nxname]["type"]=target_nxtype
  g.edges[source_nxname,target_nxname]["name"]=rel
print("done")

In [ ]:
net = Network(notebook=True, cdn_resources="remote", height="750px", width="100%", bgcolor="#222222", font_color="white")
net.from_nx(g)
net.force_atlas_2based()
net.save_graph("medical_graph.html")
display(HTML("medical_graph.html"))

In [ ]:
communities = algorithms.leiden(g)
print(f"Found {len(communities.communities)} distinct communities.")
print(f"Modularity Score: {communities.newman_girvan_modularity().score}")
for i, community_list in enumerate(communities.communities):
    print(f"Community {i}: {len(community_list)} nodes")
    print(f"  Nodes: {community_list[:5]}...") # Print first 5 nodes as preview
    if i > 4: break

In [ ]:
from pyvis.network import Network
from IPython.display import display, HTML
for community_id, members in enumerate(communities.communities):
    for node in members:
        g.nodes[node]['group'] = community_id
        g.nodes[node]['title'] = f"Community {community_id} (Member)"
net = Network(notebook=True, cdn_resources="remote", height="750px", width="100%", bgcolor="#222222", font_color="white")
net.from_nx(g)
net.force_atlas_2based(gravity=-50, central_gravity=0.01, spring_length=100, spring_strength=0.08, damping=0.4, overlap=0)
net.show_buttons(filter_=['physics'])
output_filename = "medical_clusters.html"
net.save_graph(output_filename)
display(HTML(filename=output_filename))

In [ ]:
system_template = """You are a specialized Medical Science Communicator.
Your task is to turn raw graph data into a professional, distinct, and grammatically fluent clinical summary.

### 1. THE CONSTRAINT (CLOSED WORLD)
- You may ONLY use the entities and relationships provided in the input.
- Do NOT bring in outside medical facts (no hallucinations).
- If a relationship isn't listed, you cannot mention it.

### 2. THE FLUENCY RULES (CRITICAL FOR GOOD GRAMMAR)
- **No Robotic Lists:** Do not write "A causes B. B causes C."
- **Use Compound Sentences:** Combine related edges into fluid thoughts.
  - *Bad:* "Obesity causes Diabetes. Diabetes causes Renal Failure."
  - *Good:* "Obesity is a driver of Diabetes , which subsequently leads to Renal Failure ."
- **Vary Sentence Structure:** Use transition words (e.g., "Furthermore," "Consequently," "In contrast").


Here is the strict graph data you must convert into text.

### PART A: KNOWN ENTITIES (node_list)
{node}

### PART B: VERIFIED EDGES (relationship_list)
{relationship}

### YOUR TASK:
Write a precise, grounded clinical summary based ONLY on Part B and return only the summary as output
"""

strict_graph_prompt = PromptTemplate(
    template=system_template,
    input_variables=["node","relationship"]
)


In [ ]:
community_summary=[]
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    max_retries=2,
)
chain2= strict_graph_prompt | llm
for i,member in enumerate(communities.communities):
  community_relations=[]
  community_nodes=[]
  subgraph=g.subgraph(member)
  community_node=member
  for j in subgraph.edges:
    rel=subgraph.edges[j]["name"]
    relation=j[0]+rel+j[1]
    if relation not in community_relations:
      community_relations.append(relation)
  community_response=chain2.invoke({"node":community_node,"relationship":community_relations})
  community_summary.append(community_response.content)
  if i>4:
    break

In [ ]:
final_summary_template=PromptTemplate(
    template='''
You are a Clinical Report Editor.
You have received detailed analysis reports from distinct data clusters (communities).
Your job is to compile these into a single, cohesive Clinical Review Document.

### 1. THE "NO DATA LOSS" RULE (CRITICAL)
- You must PRESERVE all specific entity names, drug dosages, metrics, percentages, and lab values.
- Never simplify a specific medical term into a general one.
-DO NOT CREATE ANY NEW INFORMATION about drug dosages , metrics, percentages, and lab values.

### 2. THE STRUCTURING TASK
- Do not just list the summaries one by one.
### 3. TONE
- Professional, objective, and dense. Do not use "fluff" words.

Here are the raw reports from the graph communities.
Combine them into one structured report.

### RAW REPORTS:
{community_summaries_text}

### FINAL CLINICAL REVIEW:
    ''',
    input_variables=["community_summaries_text"]
)
chain3= final_summary_template | llm
final_response=chain3.invoke({"community_summaries_text":community_summary})

In [ ]:
print(final_response.content)

#CHAIN


In [ ]:
!pip install langchain
!pip install langchain_text_splitters
!pip install openai
!pip install -U "langchain[google-genai]"
!pip install pydantic
!pip install langchain-groq
!pip install neo4j
!pip install networkx
!pip install pyvis
!pip install cdlib
!pip install leidenalg

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from groq import RateLimitError
import json
from google.colab import runtime
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chat_models import init_chat_model
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel,Field
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq
from neo4j import GraphDatabase
import networkx as nx
import matplotlib.pyplot as plt
from pyvis.network import Network
from IPython.display import display, HTML
from cdlib import algorithms
from typing import List, Literal
import time
from tqdm import tqdm # Progress bar
from langchain_core.output_parsers import PydanticOutputParser
import sys
from google.colab import drive
import pandas as pd
URI=""
AUTH=""
groq_key=[]
os.environ["GROQ_API_KEY"]=groq_key[current_key]
drive.mount('/content/drive')


In [ ]:
def chunked(text):
  splitter = RecursiveCharacterTextSplitter(
      chunk_size=1000,
      chunk_overlap=200,
      separators=["\n\n", "\n", " ", ""])
  chunks=splitter.split_text(text)
  return chunks
EntityType = Literal[
    # CLASS A: BIOLOGICAL AGENTS
    "PATHOGEN",          # Viruses, Bacteria, Fungi (e.g., H3N2, E. coli)
    "HOST",              # Patients, Animals, Demographics (e.g., Refugee patients, Children, Ferrets)

    # CLASS B: MOLECULAR REALITY (Crucial for your data)
    "GENETIC_ENTITY",    # Genes, RNA, DNA, Primers, Amplicons (e.g., HA Gene, PCR Primers)
    "MOLECULAR_VARIANT", # Mutations, Substitutions, Strains, Isolates (e.g., K145N, Nepal Isolates)
    "CHEMICAL",          # Drugs, Proteins, Reagents, Antibodies (e.g., Lisinopril, HA Protein)

    # CLASS C: CLINICAL REALITY
    "CONDITION",         # Diseases, Symptoms, Syndromes (e.g., Influenza, Fever)
    "PHENOMENON",        # Biological processes (e.g., Genetic Drift, Antigenic Variation)

    # CLASS D: INTERVENTION & CONTEXT
    "PROCEDURE",         # Tests, Assays, Sequencing, Vaccinations (e.g., PCR, HI Assay)
    "DEVICE",            # Machines, Kits (e.g., Magnapure LX)
    "LOCATION",          # Geographic or Anatomical (e.g., Nepal, Throat)
    "METRIC",            # Measurements, Titers (e.g., 1:320, 4-fold increase)
    "CELL",
    "ANATOMY",          # Added these 2
    "OTHER"              # Strict fallback
]

RelationType = Literal[
    "CAUSES",            # PATHOGEN/VARIANT -> CONDITION (H3N2 causes Influenza)
    "TREATS",            # CHEMICAL/PROCEDURE -> CONDITION (Tamiflu treats Influenza)
    "PREVENTS",          # VACCINE/CHEMICAL -> CONDITION (Vaccine prevents Flu)

    "EXHIBITS",          # PATHOGEN -> VARIANT (H3N2 exhibits K145N mutation)
    "DETECTS",           # PROCEDURE -> PATHOGEN/GENETIC_ENTITY (PCR detects RNA)
    "TARGETS",           # CHEMICAL -> GENETIC_ENTITY/PATHOGEN (Primers target HA Gene)

    "ORIGINATES_FROM",   # VARIANT -> LOCATION/HOST (Nepal Isolates originate from Nepal)
    "DIFFERS_FROM",      # VARIANT -> VARIANT (Strain A differs from Strain B)

    "HAS_CONTEXT",       # ENTITY -> LOCATION/METRIC/TIME (General linker)
    "ASSOCIATED_WITH",    # Fallback for weak correlations
    "INHIBITS",
    "STIMULATES",
    "PRODUCES",
    "CONTAINS",
    "MEASURES"
]

class Entity(BaseModel):
    name: str = Field(description="The precise scientific name. For 'Nepal Isolates', extract 'Isolates' as the name and 'Nepal' as a separate Location node.")
    type: EntityType = Field(description="The strict scientific category.If unsure of the category, STRICTLY use 'OTHER'.")

class Relation(BaseModel):
    source_entity: Entity = Field(description="The subject of the relation")
    relation_type: RelationType = Field(description="The strict scientific verb connecting them.If the relationship is complex, you MUST select 'ASSOCIATED_WITH'.")
    target_entity: Entity = Field(description="The object of the relation")
    evidence: str = Field(description="The EXACT quote from the text supporting this.")
    confidence: float = Field(description="Confidence score (0.0-1.0).")

class MedicalGraph(BaseModel):
    relations: List[Relation]




system_prompt = """You are an expert Medical Knowledge Graph Engineer.
Your goal is to extract a highly precise, scientifically accurate Knowledge Graph from the provided text.

### CORE INSTRUCTION: DECONSTRUCTION OVER LABELING
Medical text often compresses complex relationships into single noun phrases.
Do NOT label complex phrases as single nodes. Instead, DECONSTRUCT them into atomic components.

Examples of Deconstruction:
- "Metastatic Lung Cancer" -> "Lung Cancer" (CONDITION) + "Metastasis" (PHENOMENON).
- "Nepal Isolates" -> "Isolates" (MOLECULAR_VARIANT) + "Nepal" (LOCATION).
- "PCR Primers" -> "Primers" (GENETIC_ENTITY) + "PCR" (PROCEDURE).

### STRICT ENTITY GUIDELINES
1. PATHOGEN: Only the active infectious agent (e.g., "H3N2", "S. aureus"). Do not confuse with the Disease.
2. CONDITION: The passive disease or symptom (e.g., "Influenza", "Pneumonia").
3. CHEMICAL: Includes drugs, proteins, and lab reagents.
4. MOLECULAR_VARIANT: Use this for specific strains, mutations, or receptor statuses (e.g., "Delta Variant", "HER2+").
5. GENETIC_ENTITY: Use this for Genes, RNA, DNA, and Primers.
6. LOCATION: Use this for both Geography (Countries) and Anatomy (Body parts).

### RELATIONSHIP USAGE RULES
- Use 'EXHIBITS' to link a Subject to its Properties or Variants (e.g., Virus -> EXHIBITS -> Mutation).
- Use 'TARGETS' for Mechanism of Action (e.g., Drug -> TARGETS -> Protein).
- Use 'ORIGINATES_FROM' to link samples/variants to their source (e.g., Tumor -> ORIGINATES_FROM -> Tissue).
- Use 'HAS_CONTEXT' to link Entities to Metrics, Time, or Locations (e.g., Patient -> HAS_CONTEXT -> Age 60).
- Use 'DETECTS' for Diagnostic tools finding a Condition or Pathogen.

### OUTPUT LOGIC
- If the text mentions "Resistance to Tamiflu", extract:
  (Tamiflu: CHEMICAL) -[TREATS]-> (Influenza: CONDITION)
  (Influenza: CONDITION) -[EXHIBITS]-> (Resistance: PHENOMENON)
  (Resistance: PHENOMENON) -[ASSOCIATED_WITH]-> (Tamiflu: CHEMICAL)

Analyze the text deeply. Bias towards granular, scientifically precise nodes.
"""

few_shot_examples = """
Input: "Patient diagnosed with hypertension. Prescribed Lisinopril 10mg."
Output: [
  {"source": "Lisinopril", "relation": "TREATS", "target": "Hypertension", "evidence": "Prescribed Lisinopril 10mg"}
]

Input: "Patient denies chest pain but reports shortness of breath."
Output: [
  {"source": "Patient", "relation": "ASSOCIATED_WITH", "target": "Shortness of Breath", "evidence": "reports shortness of breath"}
  // Note: We deliberately ignored Chest Pain because it was denied.
]
"""

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "Here are some examples of correct extraction: {examples}"),
    ("human", "Analyze this clinical note: {text}")
])



merge_query="""
MERGE (e:Entity{name:$name1})
ON CREATE
  SET e.type=$type1

MERGE (f:Entity{name:$name2})
ON CREATE
  SET f.type=$type2

MERGE (e)-[r:RELATIONSHIP]->(f)
"""
relationships=[
    "CAUSES",            # PATHOGEN/VARIANT -> CONDITION (H3N2 causes Influenza)
    "TREATS",            # CHEMICAL/PROCEDURE -> CONDITION (Tamiflu treats Influenza)
    "PREVENTS",          # VACCINE/CHEMICAL -> CONDITION (Vaccine prevents Flu)

    "EXHIBITS",          # PATHOGEN -> VARIANT (H3N2 exhibits K145N mutation)
    "DETECTS",           # PROCEDURE -> PATHOGEN/GENETIC_ENTITY (PCR detects RNA)
    "TARGETS",           # CHEMICAL -> GENETIC_ENTITY/PATHOGEN (Primers target HA Gene)

    "ORIGINATES_FROM",   # VARIANT -> LOCATION/HOST (Nepal Isolates originate from Nepal)
    "DIFFERS_FROM",      # VARIANT -> VARIANT (Strain A differs from Strain B)

    "HAS_CONTEXT",       # ENTITY -> LOCATION/METRIC/TIME (General linker)
    "ASSOCIATED_WITH",    # Fallback for weak correlations
    "INHIBITS",
    "STIMULATES",
    "PRODUCES",
    "CONTAINS",
    "MEASURES"
]

def add_neo4j(final_results,merge_query,URI,AUTH):
  x=json.dumps(final_results)
  x=json.loads(x)
  with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()
    print("Connection established.")
    #record,summary,key=driver.execute_query("MATCH (p:Person) RETURN p.name",database_="neo4j")  ---------------->>>> for read operation
    record,summary,key=driver.execute_query("MATCH(n) DETACH DELETE n",database_="neo4j")
    for i in x:
      for j in i['relations']:
        source_name=j['source_entity'].get('name')
        source_type=j['source_entity'].get('type')
        target_name=j['target_entity'].get('name')
        target_type=j['target_entity'].get('type')
        relationship=j['relation_type']
        if not (source_name and target_name and relationship):
                continue
        if relationship in relationships:
          final_query=merge_query.replace("RELATIONSHIP",relationship)
          summary=driver.execute_query(final_query,name1=source_name,type1=source_type,name2=target_name,type2=target_type,database_="neo4j")
        else:
          print("----LLM HALLUCINATED AND NOT FOLLOWED ORDERS-----------------------------------------------------")


def communties_extracton(g,URI,AUTH):
  with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()
    print("Connection established.")
    records,summary,key=driver.execute_query("MATCH (n:Entity)-[r]->(m:Entity) RETURN n,r,m")
  for record in records:
    x=record.data()
    source=x.get("n")
    source_nxname=source["name"]
    source_nxtype=source["type"]
    rel=x.get("r")
    rel=rel[1]
    target=x.get("m")
    target_nxname=target["name"]
    target_nxtype=target["type"]
    g.add_node(source_nxname)
    g.add_node(target_nxname)
    g.add_edge(source_nxname,target_nxname)
    g.nodes[source_nxname]["type"]=source_nxtype
    g.nodes[target_nxname]["type"]=target_nxtype
    g.edges[source_nxname,target_nxname]["name"]=rel
  print("done")
  communities = algorithms.leiden(g)
  return communities


system_template = """You are a specialized Medical Science Communicator.
Your task is to turn raw graph data into a professional, distinct, and grammatically fluent clinical summary.

### 1. THE CONSTRAINT (CLOSED WORLD)
- You may ONLY use the entities and relationships provided in the input.
- Do NOT bring in outside medical facts (no hallucinations).
- If a relationship isn't listed, you cannot mention it.

### 2. THE FLUENCY RULES (CRITICAL FOR GOOD GRAMMAR)
- **No Robotic Lists:** Do not write "A causes B. B causes C."
- **Use Compound Sentences:** Combine related edges into fluid thoughts.
  - *Bad:* "Obesity causes Diabetes. Diabetes causes Renal Failure."
  - *Good:* "Obesity is a driver of Diabetes , which subsequently leads to Renal Failure ."
- **Vary Sentence Structure:** Use transition words (e.g., "Furthermore," "Consequently," "In contrast").


Here is the strict graph data you must convert into text.

### PART A: KNOWN ENTITIES (node_list)
{node}

### PART B: VERIFIED EDGES (relationship_list)
{relationship}

### YOUR TASK:
Write a precise, grounded clinical summary based ONLY on Part B and return only the summary as output
"""

strict_graph_prompt = PromptTemplate(
    template=system_template,
    input_variables=["node","relationship"]
)



def five_seasons(g,strict_graph_prompt,communities):
  community_summary=[]
  llm = ChatGroq(
      model="llama-3.3-70b-versatile",
      temperature=0,
      max_retries=2,
  )
  chain2= strict_graph_prompt | llm
  for i,member in enumerate(communities.communities):
    community_relations=[]
    community_nodes=[]
    subgraph=g.subgraph(member)
    community_node=member
    for j in subgraph.edges:
      rel=subgraph.edges[j]["name"]
      relation=j[0]+" "+rel+" "+j[1]
      if relation not in community_relations:
        community_relations.append(relation)
    community_response=chain2.invoke({"node":community_node,"relationship":community_relations})
    community_summary.append(community_response.content)
    if i>4:
      break
  return community_summary

final_summary_template=PromptTemplate(
    template='''
You are a Clinical Report Editor.
You have received detailed analysis reports from distinct data clusters (communities).
Your job is to compile these into a single, cohesive Clinical Review Document.

### 1. THE "NO DATA LOSS" RULE (CRITICAL)
- You must PRESERVE all specific entity names, drug dosages, metrics, percentages, and lab values.
- Never simplify a specific medical term into a general one.
-DO NOT CREATE ANY NEW INFORMATION about drug dosages , metrics, percentages, and lab values.

### 2. THE STRUCTURING TASK
- Do not just list the summaries one by one.
### 3. TONE
- Professional, objective, and dense. Do not use "fluff" words.

Here are the raw reports from the graph communities.
Combine them into one structured report.

### RAW REPORTS:
{community_summaries_text}

### FINAL CLINICAL REVIEW(RETURN ONLY THE SUMMARY):
    ''',
    input_variables=["community_summaries_text"]
)

In [ ]:
### pydantic fail logic

import json
import re
def extract_relations_from_error(e):
    failed_gen = None
    if hasattr(e, "error") and isinstance(e.error, dict):
        failed_gen = e.error.get("failed_generation")
    if failed_gen is None and hasattr(e, "body"):
        body = e.body
        if isinstance(body, dict):
            failed_gen = body.get("error", {}).get("failed_generation")
    if failed_gen is None and e.args:
        arg0 = e.args[0]
        if isinstance(arg0, dict):
            failed_gen = arg0.get("error", {}).get("failed_generation")
    if failed_gen is None:
        text = str(e)
        match = re.search(
            r"<function=MedicalGraph>(.*?)</function>",
            text,
            re.DOTALL
        )
        if match:
            failed_gen = match.group(1)
    if failed_gen is None:
        raise ValueError("failed_generation not found anywhere")
    start = failed_gen.find("{")
    end = failed_gen.rfind("}") + 1
    json_str = failed_gen[start:end]
    data = json.loads(json_str)
    return [
        {
            "source_entity": r["source_entity"]["name"],
            "source_type": r["source_entity"].get("type", "OTHER"),
            "relation": r["relation_type"],
            "target_entity": r["target_entity"]["name"],
            "target_type": r["target_entity"].get("type", "OTHER"),
            "evidence": r.get("evidence", ""),
            "confidence": r.get("confidence")
        }
        for r in data.get("relations", [])
    ]
VALID_RELATIONS = {
    "CAUSES",            # PATHOGEN/VARIANT -> CONDITION
    "TREATS",            # CHEMICAL/PROCEDURE -> CONDITION
    "PREVENTS",          # VACCINE/CHEMICAL -> CONDITION
    "EXHIBITS",          # PATHOGEN -> VARIANT
    "DETECTS",           # PROCEDURE -> PATHOGEN/GENETIC_ENTITY
    "TARGETS",           # CHEMICAL -> GENETIC_ENTITY/PATHOGEN
    "ORIGINATES_FROM",   # VARIANT -> LOCATION/HOST
    "DIFFERS_FROM",      # VARIANT -> VARIANT
    "HAS_CONTEXT",       # ENTITY -> LOCATION/METRIC/TIME
    "ASSOCIATED_WITH",   # Fallback (weak correlation)
    "INHIBITS",
    "STIMULATES",
    "PRODUCES",
    "CONTAINS",
    "MEASURES"
}
def normalize_relation(relation: str) -> str:
    if relation in VALID_RELATIONS:
        return relation
    return "ASSOCIATED_WITH"
def normalize_relations(relations):
    normalized = []
    for r in relations:
        original = r["relation"]
        fixed = normalize_relation(original)
        if original != fixed:
            print(f"[RELATION FIX] {original} → {fixed}") ###delete this if want to
        r["relation"] = fixed
        normalized.append(r)
    return normalized
def convert_relations_to_canonical(relations):
    canonical_block = {"relations": []}
    for r in relations:
        canonical_block["relations"].append({
            "source_entity": {
                "name": r["source_entity"],
                "type": r.get("source_type", "OTHER")
            },
            "relation_type": r["relation"],
            "target_entity": {
                "name": r["target_entity"],
                "type": r.get("target_type", "OTHER")
            },
            "evidence": r.get("evidence", ""),
            "confidence": float(r.get("confidence", 0.7))
        })
    return canonical_block

In [ ]:
current_abstract=60 ### change this accordingly


folder_path = '/content/drive/MyDrive/Graph_Rag_Testing'
file_name='TESTING.xlsx'
if os.path.exists(folder_path):
  file_path=os.path.join(folder_path,file_name)
  df=pd.read_excel(file_path)

while current_abstract < len(df):
  chunks=chunked(df.iloc[current_abstract,1])
  llm = ChatGroq(model="llama-3.3-70b-versatile",temperature=0,max_retries=2,)

  structured_llm = llm.with_structured_output(MedicalGraph)

  chain = prompt_template | structured_llm
  final_results = []
  errors = []

  print(f"Starting extraction on {len(chunks)} chunks...")

  for i, chunk in tqdm(enumerate(chunks), total=len(chunks)):
    while True:
        try:
            result = chain.invoke({
                "examples": few_shot_examples,
                "text": chunk
            })

            final_results.append(result.model_dump())
            break

        except RateLimitError:
          if current_key<len(groq_key)-1:
            current_key+=1
            print("KEY CHANGED-----------------------------------------------,current key:",current_key)
          else:
            runtime.unassign()
            time.sleep(10)
            sys.exit()
          os.environ["GROQ_API_KEY"]=groq_key[current_key]
          llm = ChatGroq(model="llama-3.3-70b-versatile",temperature=0,max_retries=2,)
          structured_llm = llm.with_structured_output(MedicalGraph)
          chain = prompt_template | structured_llm

        except Exception as e:
            print(f"Error on chunk {i}: {e}")
            errors.append({"chunk_index": i, "error": str(e)})
            relations = extract_relations_from_error(e)
            relations = normalize_relations(relations)
            canonical = convert_relations_to_canonical(relations)
            final_results.append(canonical)
            print("Successful")
            break
        time.sleep(4)

  add_neo4j(final_results,merge_query,URI,AUTH)
  g=nx.Graph()
  communities=communties_extracton(g,URI,AUTH)
  while True:
    try:
      community_summary=five_seasons(g,strict_graph_prompt,communities)
      break
    except RateLimitError:
      if current_key<len(groq_key)-1:
        current_key+=1
        print("KEY CHANGED-----------------------------------------------,current key:",current_key)
      else:
        runtime.unassign()
        time.sleep(10)
        sys.exit()
      os.environ["GROQ_API_KEY"]=groq_key[current_key]
  while True:
    try:
      llm = ChatGroq(model="llama-3.3-70b-versatile",temperature=0,max_retries=2,)
      chain3= final_summary_template | llm
      final_response=chain3.invoke({"community_summaries_text":community_summary})
      break
    except RateLimitError:
      if current_key<len(groq_key)-1:
        current_key+=1
        print("KEY CHANGED------------------------&&&&&&^^^^^^^%%%%%%%%%%%-----------------------,current key:",current_key)
      else:
        runtime.unassign()
        time.sleep(10)
        sys.exit()
      os.environ["GROQ_API_KEY"]=groq_key[current_key]
  print(final_response.content)
  df.at[current_abstract, 'summary']=final_response.content
  df.to_excel(file_path,index=False)
  current_abstract+=1
  print("UPDATED SUCESSFULLY------------------------$$$$$$$$$$$$$$$%%%%%%%%%%%%%%%%%%%%-----------------------------")
